# RAG with and without reranking

See how reranking changes the selected passages and whether that changes the generated answer.

## One question, two RAG paths

Compare both paths on **“What is the cause of COVID-19?”** Each path selects five passages from the same 30 candidates and uses the same prompt template and generator settings.

| Component | Model or setup |
| --- | --- |
| Embedding | `qwen/qwen3-embedding-4b` · 2,560 dimensions |
| Reranker | `qwen/qwen3-reranker-8b` |
| Generator | `qwen/qwen3-30b-a3b-instruct-2507` |
| Dataset | MS MARCO v2.1 validation (`microsoft/ms_marco`) |
| Pipeline | Haystack: query embedding → retrieve 30 passages → select five, or rerank then select five → generate answer |


## Setup and saved execution

Use the project's pinned environment and dependency lock. These saved outputs show the recorded Qwen execution; no rerun is needed to read them. Set `NOTEBOOK_MODE=replay` before starting the kernel for deliberate replay. Execute mode can make missing paid calls only with the required action settings and `OPENROUTER_API_KEY`.

The index covers the frozen **152,576-passage validation subset**, not the full validation corpus. Document embedding was stopped after this snapshot was established. The successful retrieval and answer outputs are preserved below. Exact Python and package versions should be checked against the repository lock and recorded environment metadata; this cleanup does not upgrade dependencies.


In [1]:
# Qwen/OpenRouter components used by the actual Haystack pipelines below.
from pathlib import Path
from copy import deepcopy
import sys, json, os, re
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from haystack import Pipeline, Document, component
from haystack.components.builders import PromptBuilder

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/execution.json').is_file())
sys.path.insert(0, str(PROJECT))
from shared_scoring.notebook_runtime import NotebookSession, StudyNotReady
from shared_scoring.fresh_retrieval import RecordedOpenRouterClient, fingerprint, normalized_vectors
from shared_scoring.io import read_json, write_json
session = NotebookSession(PROJECT)
MODE = session.mode
execution = session.settings
retrieval_settings = execution['retrieval']
ARTIFACTS = session.retrieval_directory()
settings = dict(embedding_model=retrieval_settings['embedding_model'],
                reranker_model=retrieval_settings['reranker_model'],
                generator_model='qwen/qwen3-30b-a3b-instruct-2507',
                embedding_dimensions=retrieval_settings['dimensions'],
                dataset_revision=execution['dataset']['revision'],
                question='What is the cause of COVID-19?', top_candidates=30,
                top_context=5, temperature=0.7)
DEMO_CACHE = execution['runtime_directory'] / 'demo' / fingerprint(settings)[:20]

class DemoClient(RecordedOpenRouterClient):
    def request(self, stage, url, payload, validate=None, cache_key=None):
        contract = dict(stage=stage, url=url, payload=payload)
        if cache_key is not None:
            contract['cache_key'] = cache_key
        attempts = list((self.directory / fingerprint(contract)).glob('attempt-*.json'))
        if not attempts:
            session.enable_calls()  # Tests first; explicit credentials only on cache miss.
        return super().request(stage, url, payload, validate, cache_key)

api = DemoClient(dict(retrieval_settings), DEMO_CACHE / 'requests')

@component
class QwenTextEmbedder:
    def __init__(self, client): self.client = client
    @component.output_types(embedding=list)
    def run(self, text: str):
        vectors, trace = self.client.embed([text], 'query', settings['embedding_dimensions'])
        write_json(DEMO_CACHE / 'query_embedding_trace.json', trace)
        return {'embedding': vectors[0].tolist()}

@component
class PrecomputedEmbeddingRetriever:
    def __init__(self, index, corpus, top_k=30):
        self.index, self.corpus, self.top_k = index, corpus, top_k
    @component.output_types(documents=list[Document])
    def run(self, query_embedding: list):
        vector = normalized_vectors([query_embedding], 1, settings['embedding_dimensions'])
        scores, indices = self.index.search(vector, self.top_k)
        if (indices < 0).any():
            raise ValueError('Retrieval returned missing corpus rows.')
        documents = []
        for rank, (position, score) in enumerate(zip(indices[0], scores[0]), 1):
            row = self.corpus.iloc[int(position)]
            documents.append(Document(id=row.passage_id, content=row.text, score=float(score),
                meta={'source_url': row.source_url, 'embedding_rank': rank, 'truncated': bool(row.truncated)}))
        write_json(DEMO_CACHE / 'candidates.json', [d.to_dict() for d in documents])
        return {'documents': documents}

@component
class SelectTopFive:
    @component.output_types(documents=list[Document])
    def run(self, documents: list[Document]):
        return {'documents': documents[:settings['top_context']]}

@component
class QwenRanker:
    def __init__(self, client): self.client = client
    @component.output_types(documents=list[Document])
    def run(self, documents: list[Document], query: str):
        rankings, trace = self.client.rerank(query, [{'text': d.content} for d in documents])
        write_json(DEMO_CACHE / 'reranking_trace.json', trace)
        selected = [documents[r['index']] for r in rankings[:settings['top_context']]]
        return {'documents': selected}

@component
class QwenGenerator:
    def __init__(self, client, arm): self.client, self.arm = client, arm
    @component.output_types(replies=list[str])
    def run(self, prompt: str):
        payload = {'model': settings['generator_model'],
                   'messages': [{'role': 'user', 'content': prompt}], 'temperature': settings['temperature']}
        previous = self.client.settings['max_attempts']
        self.client.settings['max_attempts'] = 1
        try:
            raw, trace = self.client.request('generation', retrieval_settings['api_base'] + '/chat/completions',
                                             payload, cache_key='demo-' + self.arm)
        finally:
            self.client.settings['max_attempts'] = previous
        write_json(DEMO_CACHE / (self.arm + '-generation.json'), trace)
        return {'replies': [raw['choices'][0]['message']['content']]}

def context_table(documents):
    return pd.DataFrame([{'id': d.id, 'text': d.content, 'score': d.score, **d.meta} for d in documents])

def show_arm(arm, result, selected):
    display(context_table(selected))
    display(Markdown(result['generator']['replies'][0]))
    trace = read_json(DEMO_CACHE / (arm + '-generation.json'))
    display({'request_body': trace['request_body'], 'raw_response': trace['raw_response_json'],
             'elapsed_seconds': trace['elapsed_seconds'], 'cache_hit': trace.get('cache_hit', False)})


## Dataset

The corpus uses MS MARCO v2.1 validation. The following row illustrates the dataset structure; the full dataset is not bundled.


In [3]:
if not (ARTIFACTS / 'dataset_manifest.json').is_file():
    raise StudyNotReady('The pinned local dataset has not been prepared yet.')
corpus_metadata = read_json(ARTIFACTS / 'dataset_manifest.json')
print('Validation questions:', corpus_metadata['validation_questions'])
print('Dataset revision:', settings['dataset_revision'])


Validation questions: 15585
Dataset revision: a47ee7aae8d7d466ba15f9f0bfac3b3681087b3a


Inspect a question, its passages, source URLs, support annotations and reference answers. This Rachel Carson example explains the schema; the pipeline demonstration uses COVID-19.

In `is_selected`, **1** marks a passage annotated as supporting the reference answer. These dataset labels let us evaluate retrieval coverage.


In [4]:
dataset_rows = read_json(PROJECT / 'experiments/v1-baseline/input/dataset_rows.json')
display(dataset_rows[0])


{'answers': ['Rachel Carson writes The Obligation to Endure because believes that as man tries to eliminate unwanted insects and weeds, however he is actually causing more problems by polluting the environment.'],
 'passages': {'is_selected': [0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
  'passage_text': ["Read to write - grow a love of books (even if it's a comic book) Organizing an argument into paragraphs, words [that] worked together for a common purpose Walker: Using models - read other research papers, look at MLA samples Maintain a larger perspective of the world and in your writing - stay objective Write about what fascinates you",
   'Carson subtly defers her writing in just the right writing technique for it to not be subject to an induction run rampant style which grabs the readers interest without biasing the whole article. Rachel slights the opposition very nicely Rachel Carson: “THE OBLIGATION TO ENDURE”. 4 by keeping it simple yet informative and persuasive and stays away from being 

Each candidate retains its passage text, source metadata, embedding rank and similarity score. The corpus contains deduplicated passages with stable IDs and a 1,800-character passage limit. The retrieval section below displays the thirty candidates after retrieval has completed.


In [5]:
def corpus_row_to_document(row):
    return Document(id=row['passage_id'], content=row['text'],
                    meta={'source_url': row['source_url'], 'truncated': row['truncated']})


## Query embedding and retrieval

Document embeddings and the FAISS index already exist for the frozen 152,576-passage corpus. The recorded workflow embeds the question, retrieves thirty candidates, and saves separate top-five contexts and generator responses. Successful matching requests can be reused; these saved outputs are retained without execution.


In [7]:
display(pd.DataFrame(settings.items(), columns=['Setting', 'Value']))


,Setting,Value
0,embedding_model,qwen/qwen3-embedding-4b
1,reranker_model,qwen/qwen3-reranker-8b
2,generator_model,qwen/qwen3-30b-a3b-instruct-2507
3,embedding_dimensions,2560
4,dataset_revision,a47ee7aae8d7d466ba15f9f0bfac3b3681087b3a
5,question,What is the cause of COVID-19?
6,top_candidates,30
7,top_context,5
8,temperature,0.7


The following cell shows the Haystack retrieval pipeline over the frozen **152,576 cached passages**. Its saved output contains the thirty candidates from the recorded run. A future execution requires the corresponding corpus/index and matching configuration.


In [8]:
def create_retrieval_pipeline(api, faiss_index, corpus):
    pipeline = Pipeline()
    pipeline.add_component('embedder', QwenTextEmbedder(api))
    pipeline.add_component('retriever', PrecomputedEmbeddingRetriever(faiss_index, corpus, top_k=30))
    pipeline.connect('embedder.embedding', 'retriever.query_embedding')
    return pipeline

session.require_complete_index()  # Guard: never resumes document embedding.
import faiss
faiss.omp_set_num_threads(retrieval_settings['threads'])
manifest = read_json(ARTIFACTS / 'assets_manifest.json')
corpus = pd.read_parquet(ARTIFACTS / 'corpus.parquet')
faiss_index = faiss.read_index(str(ARTIFACTS / 'index.faiss'))
faiss_index.nprobe = manifest['nprobe']
retrieval_pipeline = create_retrieval_pipeline(api, faiss_index, corpus)
retrieval_result = retrieval_pipeline.run({'embedder': {'text': settings['question']}})
frozen_candidates = retrieval_result['retriever']['documents']
display(context_table(frozen_candidates))


,id,text,score,source_url,embedding_rank,truncated
0,p_3728f219848bf793cb32,The truth that we must stare down is that cancer and infections cannot be separated from each other. Cancer—always believed to be caused by genetic cell mut...,0.491919,http://drsircus.com/cancer/is-cancer-a-fungus/,1,False
1,p_39df7d1187fcc5489493,"Bacteria are the most common cause of infective endocarditis, especially streptococci, staphylococci, and enterococci. Although fungi and viruses can cause ...",0.470086,https://quizlet.com/144209873/unit-6-chapter-23-alterations-of-cardio-function-flash-cards/,2,False
2,p_de76223c44b3e69cd180,"California is a state, so it doesn't think! :-)) Just kidding. Cancer is caused by a genetic mutation. Genetic mutations are caused by UV light (sun, sun la...",0.469425,https://answers.yahoo.com/question/index?qid=20080803190629AAlO45U,3,False
3,p_8729b7573df5124d2792,"Hand, foot, and mouth disease is generally caused by coxsackievirus A-16, which is a member of the enterovirus family. There are other types of enteroviruse...",0.463244,https://www.emedicinehealth.com/hand_foot_and_mouth_disease/article_em.htm,4,False
4,p_2c34293f7e92f17e5467,"You can also have it and not know it. Doctors call this “walking pneumonia.” If your pneumonia is caused by a bacteria or virus, you can spread it to someon...",0.461744,https://www.webmd.com/lung/understanding-pneumonia-basics,5,False
5,p_a727a7d1be3cf3c5a664,"HIV-1, which causes almost all the cases of AIDS worldwide; HIV-2, which causes an AIDS-like illness. HIV-2 infection is uncommon in North America. What cau...",0.458251,https://www.webmd.com/hiv-aids/tc/human-immunodeficiency-virus-hiv-infection-topic-overview,6,False
6,p_36c253c659a76c80373a,"In most cases, acute pancreatitis is caused by gallstones or heavy alcohol use. Other causes include medications, autoimmune disease, infections, trauma, me...",0.456779,https://www.webmd.com/digestive-disorders/digestive-diseases-pancreatitis,7,False
7,p_0ff8dfc59276423c2b92,Streptococcal infections are caused by any one of several species of Streptococcus. Different groups of these bacteria are spread in different ways—for exam...,0.456163,http://www.merckmanuals.com/home/infections/bacterial-infections/streptococcal-infections,8,False
8,p_44a4729531569405d184,"The cause of cholera is infection by the V. cholera bacteria. These bacteria were discovered in 1883. The German bacteriologist, Robert Koch (1843-1910), st...",0.453879,https://www.medicalnewstoday.com/articles/189269.php,9,False
9,p_65af6d90ff5ebade896e,A cytokine storm is a potentially fatal immune reaction in which the cytokines continue to stimulate the production of immune cells when they are no longer ...,0.453026,http://www.wellness.com/reference/allergies/cytokines,10,False


## Embedding RAG

Select embedding ranks 1–5 from the same thirty candidates. The Haystack pipeline connects this selection to the prompt builder and generator.

Inspect the shared prompt template, followed by this path’s five-passage context and generated answer. Both RAG paths use the same template and generator settings.

In [9]:
prompt = """Answer the question given the context.
Question: {{ query }}
Context:
{% for document in documents %}
    {{ document.content }}
{% endfor %}
Answer:
"""

def create_embedding_pipeline(api, prompt):
    pipeline = Pipeline()
    pipeline.add_component('select_top_five', SelectTopFive())
    pipeline.add_component('prompt_builder', PromptBuilder(template=prompt, required_variables='*'))
    pipeline.add_component('generator', QwenGenerator(api, 'baseline'))
    pipeline.connect('select_top_five.documents', 'prompt_builder.documents')
    pipeline.connect('prompt_builder.prompt', 'generator.prompt')
    return pipeline

print('Original shared prompt template:\n' + prompt)
embedding_pipeline = create_embedding_pipeline(api, prompt)


Original shared prompt template:
Answer the question given the context.
Question: {{ query }}
Context:
{% for document in documents %}
    {{ document.content }}
{% endfor %}
Answer:



In [10]:
question = settings['question']
results = embedding_pipeline.run({'select_top_five': {'documents': frozen_candidates},
                                  'prompt_builder': {'query': question}},
                                 include_outputs_from={'select_top_five'})
baseline_top_five = results['select_top_five']['documents']
show_arm('baseline', results, baseline_top_five)
write_json(DEMO_CACHE / 'baseline-result.json', {'answer': results['generator']['replies'][0],
           'selected_top5': [d.to_dict() for d in baseline_top_five]})


,id,text,score,source_url,embedding_rank,truncated
0,p_3728f219848bf793cb32,The truth that we must stare down is that cancer and infections cannot be separated from each other. Cancer—always believed to be caused by genetic cell mut...,0.491919,http://drsircus.com/cancer/is-cancer-a-fungus/,1,False
1,p_39df7d1187fcc5489493,"Bacteria are the most common cause of infective endocarditis, especially streptococci, staphylococci, and enterococci. Although fungi and viruses can cause ...",0.470086,https://quizlet.com/144209873/unit-6-chapter-23-alterations-of-cardio-function-flash-cards/,2,False
2,p_de76223c44b3e69cd180,"California is a state, so it doesn't think! :-)) Just kidding. Cancer is caused by a genetic mutation. Genetic mutations are caused by UV light (sun, sun la...",0.469425,https://answers.yahoo.com/question/index?qid=20080803190629AAlO45U,3,False
3,p_8729b7573df5124d2792,"Hand, foot, and mouth disease is generally caused by coxsackievirus A-16, which is a member of the enterovirus family. There are other types of enteroviruse...",0.463244,https://www.emedicinehealth.com/hand_foot_and_mouth_disease/article_em.htm,4,False
4,p_2c34293f7e92f17e5467,"You can also have it and not know it. Doctors call this “walking pneumonia.” If your pneumonia is caused by a bacteria or virus, you can spread it to someon...",0.461744,https://www.webmd.com/lung/understanding-pneumonia-basics,5,False


The cause of COVID-19 is a virus called SARS-CoV-2.

{'request_body': {'model': 'qwen/qwen3-30b-a3b-instruct-2507',
  'messages': [{'role': 'user',
    'content': "Answer the question given the context.\nQuestion: What is the cause of COVID-19?\nContext:\n\n    The truth that we must stare down is that cancer and infections cannot be separated from each other. Cancer—always believed to be caused by genetic cell mutations—can in reality be caused by infections from viruses, bacteria, and fungi.\n\n    Bacteria are the most common cause of infective endocarditis, especially streptococci, staphylococci, and enterococci. Although fungi and viruses can cause infective endocarditis, they are not the most frequent cause. Infective endocarditis is caused by infection with microorganisms, not an autoimmune response. A client develops inflammation of the endocardium caused by Staphylococcus aureus.\n\n    California is a state, so it doesn't think! :-)) Just kidding. Cancer is caused by a genetic mutation. Genetic mutations are caused by UV light 

## Embedding + reranking RAG

Rerank the thirty candidates and select the five highest-ranked passages. The Haystack pipeline connects that selection to the prompt builder and generator.

This path uses the same prompt template and generator settings as Embedding RAG.


In [11]:
def create_reranked_pipeline(api, prompt):
    pipeline = Pipeline()
    pipeline.add_component('reranker', QwenRanker(api))
    pipeline.add_component('prompt_builder', PromptBuilder(template=prompt, required_variables='*'))
    pipeline.add_component('generator', QwenGenerator(api, 'reranked'))
    pipeline.connect('reranker.documents', 'prompt_builder.documents')
    pipeline.connect('prompt_builder.prompt', 'generator.prompt')
    return pipeline

reranked_pipeline = create_reranked_pipeline(api, prompt)


Inspect the reranked five-passage context and generated answer. Expand the trace for the rendered prompt, request body, response, finish reason, token usage and measured timing.


In [12]:
better_results = reranked_pipeline.run({'reranker': {'documents': frozen_candidates, 'query': question},
                                        'prompt_builder': {'query': question}},
                                       include_outputs_from={'reranker'})
reranked_top_five = better_results['reranker']['documents']
show_arm('reranked', better_results, reranked_top_five)
write_json(DEMO_CACHE / 'reranked-result.json', {'answer': better_results['generator']['replies'][0],
           'selected_top5': [d.to_dict() for d in reranked_top_five]})


,id,text,score,source_url,embedding_rank,truncated
0,p_65af6d90ff5ebade896e,A cytokine storm is a potentially fatal immune reaction in which the cytokines continue to stimulate the production of immune cells when they are no longer ...,0.453026,http://www.wellness.com/reference/allergies/cytokines,10,False
1,p_6482a0495822ba20641b,Primary Cause of Influenza B . The primary cause of Influenza B is the result: of transmission of an infectious agent by another person by one or more of th...,0.448999,http://www.rightdiagnosis.com/i/influenza_b/causes.htm,15,False
2,p_696be041d02106a27d65,Myocarditis is an inflammatory disease of the heart muscle (myocardium) that can result from a variety of causes. While most cases are produced by a viral i...,0.447175,https://medical-dictionary.thefreedictionary.com/Inflammation+of+the+heart,16,False
3,p_d5fc92f8a485f1700c58,The cause is unknown. Researchers think that fibromyalgia is due to a problem with how the central nervous system processes pain. Possible causes or trigger...,0.439789,https://medlineplus.gov/ency/article/000427.htm,22,False
4,p_72663f8dc5ea39d41e04,This is mostly due to the fact that a C. albicans infection stems from the commensal population of C. albicans in the human microflora. Candidiasis is cause...,0.437836,https://microbewiki.kenyon.edu/index.php/Candida_albicans_(Pathogenesis),24,False


The cause of COVID-19 is the SARS-CoV-2 virus.

{'request_body': {'model': 'qwen/qwen3-30b-a3b-instruct-2507',
  'messages': [{'role': 'user',
    'content': 'Answer the question given the context.\nQuestion: What is the cause of COVID-19?\nContext:\n\n    A cytokine storm is a potentially fatal immune reaction in which the cytokines continue to stimulate the production of immune cells when they are no longer needed. The cause of this reaction remains unknown.\n\n    Primary Cause of Influenza B . The primary cause of Influenza B is the result: of transmission of an infectious agent by another person by one or more of the following: saliva, air, cough, fecal-oral route, surfaces, blood, needles, blood transfusions, sexual contact, mother to fetus, etc. Influenza B: Related Medical Conditions\n\n    Myocarditis is an inflammatory disease of the heart muscle (myocardium) that can result from a variety of causes. While most cases are produced by a viral infection, an inflammation of the heart muscle may also be instigated by toxins, dr

## What changed?

Both recorded answers identify SARS-CoV-2, but neither selected context mentions COVID-19 or SARS-CoV-2. This prompt does not explicitly prohibit outside knowledge, so a correct answer alone does not demonstrate evidence support or a reranking benefit.

The selected passages changed; the core answer did not. This optional example motivates the evidence-only prompts and separate retrieval, reference-overlap, and faithfulness evaluations in the experiments.


In [13]:
rows = []
for arm, result, selected in [('baseline', results, baseline_top_five), ('reranked', better_results, reranked_top_five)]:
    answer = result['generator']['replies'][0]
    context = ' '.join(d.content for d in selected).casefold()
    trace = read_json(DEMO_CACHE / (arm + '-generation.json'))
    choice = trace['raw_response_json']['choices'][0]
    rows.append({'rag_type': arm, 'answer': answer, 'names_sars_cov_2': 'sars-cov-2' in answer.casefold(),
                 'finish_reason': choice.get('finish_reason'), 'output_truncated': choice.get('finish_reason') == 'length',
                 'empty_answer': not answer.strip(), 'context_mentions_covid_19': 'covid-19' in context,
                 'context_mentions_sars_cov_2': 'sars-cov-2' in context,
                 'generation_successful_request_seconds': trace['elapsed_seconds']})
comparison = pd.DataFrame(rows)
display(comparison)
assert len(frozen_candidates) == settings['top_candidates']
assert len(baseline_top_five) == len(reranked_top_five) == settings['top_context']
assert [d.id for d in baseline_top_five] == [d.id for d in frozen_candidates[:settings['top_context']]]
print('Both paths identify SARS-CoV-2:', bool(comparison.names_sars_cov_2.all()))
print('Both paths produced identical wording:', comparison.answer.nunique() == 1)
write_json(DEMO_CACHE / 'comparison.json', rows)


,rag_type,answer,names_sars_cov_2,finish_reason,output_truncated,empty_answer,context_mentions_covid_19,context_mentions_sars_cov_2,generation_successful_request_seconds
0,baseline,The cause of COVID-19 is a virus called SARS-CoV-2.,True,stop,False,False,False,False,1.204402
1,reranked,The cause of COVID-19 is the SARS-CoV-2 virus.,True,stop,False,False,False,False,1.396032


Both paths identify SARS-CoV-2: True
Both paths produced identical wording: False
